# Checkpointer——对话记忆

Checkpointer 是 LangChain 的对话记忆机制。它为 Agent 分配一个 `thread_id`，在多次调用之间保持对话上下文。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## 为什么需要 Checkpointer

Agent 默认是无状态的——每次 `invoke()` 都是全新对话。Checkpointer 通过保存和恢复 Agent 状态来实现记忆。

## 使用 Checkpointer

创建 Checkpointer 后传给 `create_agent()` 的 `checkpointer` 参数。

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

# 创建内存 Checkpointer（生产环境建议用数据库实现）
checkpointer = MemorySaver()

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
    system_prompt="你是菜鸟教程 RUNOOB 的助手。",
)

# 通过 config 传入 thread_id
config = {"configurable": {"thread_id": "conversation-001"}}

# 第一轮对话
result = agent.invoke({"messages": [HumanMessage(content="你好，我叫小明")]}, config)
print(f"第一轮: {result['messages'][-1].content}")

# 第二轮对话（Agent 记得前文）
result = agent.invoke({"messages": [HumanMessage(content="我叫什么名字？")]}, config)
print(f"第二轮: {result['messages'][-1].content}")

第一轮: 你好，小明！我是菜鸟教程的助手，很高兴认识你。无论你是想学习编程、Web开发、数据库，还是其他技术知识，都可以问我。有什么问题尽管说，我会尽力帮你解答！😊
第二轮: 你叫小明呀！刚才你告诉我的，我可不会忘记～ 😊


同一 `thread_id` 的多次对话会自动关联，Agent 能记住之前的内容。不同 `thread_id` 的对话互相隔离。

## 多个独立对话

不同的 `thread_id` 对应完全隔离的对话）。

In [3]:
config1 = {"configurable": {"thread_id": "user-001"}}
config2 = {"configurable": {"thread_id": "user-002"}}

# user-001：第一轮
result = agent.invoke({"messages": [HumanMessage(content="我叫小明")]}, config1)

# user-002：第一轮（独立空间，不共享记忆）
result = agent.invoke({"messages": [HumanMessage(content="我叫小红")]}, config2)

# user-001：第二轮（记得自己叫小明，不知道小红）
result = agent.invoke({"messages": [HumanMessage(content="我叫什么？")]}, config1)
print(result['messages'][-1].content)

你之前提到过，你叫小明。对吗？😊


## 线程 ID 的命名约定

- `user-001` 按用户隔离
- `session-abc` 按会话隔离
- `order-123` 按业务隔离

**术语：**

- **Checkpointer**：对话记忆组件，保存 Agent 状态
- **thread_id**：对话线程 ID，关联同一次多轮对话
- **MemorySaver**：内存中的 Checkpointer 实现（生产环境用数据库）